# 🌍 Earth Field Analysis
## Layer 4 – Ionosphere

| Layer | Name | Status |
|-------|------|--------|
| 0 | External Cosmic Drivers | ✅ `layer0_state.json` |
| 1 | Planetary Body | ✅ `layer1_state.json` |
| 2 | Surface / Oceans / Land | ✅ `layer2_state.json` |
| 3 | Atmosphere / Weather / Thunderstorms | ✅ `layer3_state.json` |
| **4** | **Ionosphere** | **← this layer** |
| 5 | Global Electric Circuit | ⬜ |
| 6 | Resonance Field / Schumann | ⬜ |
| 7 | Earth Field State Engine | ⬜ |

> **Core function:** Upper boundary of the Earth-Ionosphere Cavity – responds to solar and space weather, modifies the propagation of electromagnetic waves.
>
> **Core question:** How does the ionosphere alter the resonance conditions of the planetary field system?

In [1]:
import warnings; warnings.filterwarnings('ignore')
import datetime, json, math, re, requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

print(f'Analysis date: {datetime.date.today()}')

# Load all previous layers
context = {}
for n in [0, 1, 2, 3]:
    try:
        with open(f'layer{n}_test_state.json', encoding='utf-8') as f:
            context[n] = json.load(f)
        print(f'  Layer {n}: {context[n]["level"].upper():8}  Score={context[n]["score"]}')
    except FileNotFoundError:
        print(f'  Layer {n}: not found')
        context[n] = None

layer0 = context.get(0)
layer1 = context.get(1)
layer2 = context.get(2)
layer3 = context.get(3)

# Relevant upstream flags
l0_kp    = layer0.get('raw_values',{}).get('Kp_index',{}).get('value') if layer0 else None
l0_f107  = layer0.get('raw_values',{}).get('F10.7_sfu',{}).get('value') if layer0 else None
l0_bz    = layer0.get('raw_values',{}).get('IMF_Bz_nT',{}).get('value') if layer0 else None
l3_xray  = layer3.get('raw_values',{}).get('xray_context') if layer3 else None
l3_schumann = layer3.get('raw_values',{}).get('schumann_potential') if layer3 else None
print(f'\n  L0: Kp={l0_kp}  F10.7={l0_f107}  Bz={l0_bz}')
print(f'  L3: X-Ray={l3_xray}  Schumann-Pot={l3_schumann}')

Analysis date: 2026-06-27
  Layer 0: not found
  Layer 1: not found
  Layer 2: not found
  Layer 3: not found

  L0: Kp=None  F10.7=None  Bz=None
  L3: X-Ray=None  Schumann-Pot=None


---
## 1. System Structure: Ionospheric Layers & Drivers

In [2]:
# Ionospheric Layer Profile as an Altitude Diagram
layers_iono = [
    {'name': 'D-Layer',  'h_lo': 60,  'h_hi': 90,  'color': '#378ADD', 'note': '60–90 km | Daytime HF absorption'},
    {'name': 'E-Layer',  'h_lo': 90,  'h_hi': 150, 'color': '#2ecc71', 'note': '90–150 km | Sporadic-E, reflection'},
    {'name': 'F1-Layer', 'h_lo': 150, 'h_hi': 200, 'color': '#F2A623', 'note': '150–200 km | Daytime, weak'},
    {'name': 'F2-Layer', 'h_lo': 200, 'h_hi': 500, 'color': '#E85D24', 'note': '200–500 km | Main reflection, NmF2'},
    {'name': 'Plasmasphere','h_lo': 500,'h_hi': 700,  'color': '#7F77DD', 'note': '>500 km | Plasma, GEC upper boundary'},
]

fig = go.Figure()
for l in layers_iono:
    h_mid = (l['h_lo'] + l['h_hi']) / 2
    fig.add_trace(go.Bar(
        x=[l['h_hi'] - l['h_lo']],
        y=[l['name']],
        base=l['h_lo'],
        orientation='h',
        marker_color=l['color'], opacity=0.78,
        text=l['note'],
        textposition='inside',
        textfont=dict(size=10, color='white'),
        showlegend=False,
        hovertemplate=l['note'] + '<extra></extra>'
    ))

# External Driver Arrows as Annotations
for txt, x, col in [
    ('☀️ Solar UV/X-Ray → ionizes F2/E', 480, '#E85D24'),
    ('🌙 Night → D-Layer dissolves', 350, '#378ADD'),
    ('⚡ Kp/CME → F-Layer disturbance', 220, '#7F77DD'),
    ('⛈️ Thunderstorms → TIE/Sprites (D/E)', 90, '#639922'),
]:
    fig.add_annotation(
        x=x, y=4.6, text=txt, showarrow=False,
        font=dict(size=9, color=col),
        xanchor='center'
    )

# Schumann Cavity Marking
fig.add_vline(x=90, line_dash='dot', line_color='#F2A623', line_width=1.5,
              annotation_text='Schumann Cavity upper boundary (~90 km)',
              annotation_position='top right',
              annotation_font=dict(size=9, color='#F2A623'))

# Layer Context
ctx = []
if layer0: ctx.append(f'L0: Kp={l0_kp}  F10.7={l0_f107}')
if layer3 and l3_xray: ctx.append(f'X-Ray: {l3_xray.get("class","–")}-class')
if ctx:
    fig.add_annotation(x=350, y=-0.7, text='  |  '.join(ctx), showarrow=False,
                       font=dict(size=10, color='#888780'), xanchor='center')

fig.update_layout(
    title=dict(text='Ionospheric Layers – Height Profile & Drivers', font=dict(size=15)),
    xaxis=dict(title='Altitude [km]', range=[0, 720]),
    yaxis=dict(title='Layer'),
    height=400, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=80, r=30, t=60, b=60),
    barmode='overlay'
)
fig.show()

---
## 2. Fetch Real-Time Data

In [3]:
# ============================================================
# REAL-TIME DATA – FIVE SOURCES
# 1) NOAA SWPC  – Kp-Index (geomagnetic disturbance)
# 2) NOAA SWPC  – GOES X-Ray (Solar UV proxy / F2 ionization)
# 3) NOAA SWPC  – Solar radio flux F10.7 (ionization level)
# 4) NOAA SWPC  – Ionospheric disturbance bulletin
# 5) NASA/CDDIS – TEC proxy via IONEX (if available)
# ============================================================

raw = {}

# --- 1) Kp-Index ---
try:
    r = requests.get('https://services.swpc.noaa.gov/json/planetary_k_index_1m.json', timeout=12)
    r.raise_for_status()
    raw['kp'] = r.json()
    print(f'  OK NOAA Kp             {len(raw["kp"]):>5} entries')
except Exception as e:
    print(f'  ERR NOAA Kp            {str(e)[:70]}')
    raw['kp'] = None

# --- 2) GOES X-Ray ---
try:
    r = requests.get('https://services.swpc.noaa.gov/json/goes/primary/xrays-1-day.json', timeout=12)
    r.raise_for_status()
    raw['xray'] = r.json()
    print(f'  OK GOES X-Ray          {len(raw["xray"]):>5} entries')
except Exception as e:
    print(f'  ERR GOES X-Ray         {str(e)[:70]}')
    raw['xray'] = None

# --- 3) F10.7 Solar radio flux (current month) ---
try:
    r = requests.get(
        'https://services.swpc.noaa.gov/json/solar-cycle/observed-solar-cycle-indices.json',
        timeout=12)
    r.raise_for_status()
    raw['f107'] = r.json()
    last = raw['f107'][-1]
    print(f'  OK F10.7               aktuell {last.get("f10.7","–")} sfu  ({last.get("time_tag","–")[:7]})')
except Exception as e:
    print(f'  ERR F10.7              {str(e)[:70]}')
    raw['f107'] = None

# --- 4) NOAA SWPC Ionospheric disturbance bulletin ---
try:
    r = requests.get('https://services.swpc.noaa.gov/text/wwv.txt', timeout=12)
    r.raise_for_status()
    raw['wwv'] = r.text
    lines = [l for l in r.text.splitlines() if l.strip()]
    print(f'  OK NOAA WWV Bulletin   {len(lines):>5} lines')
except Exception as e:
    print(f'  ERR NOAA WWV           {str(e)[:70]}')
    raw['wwv'] = None

# --- 5) NOAA SWPC Alerts (ionospheric warnings) ---
try:
    r = requests.get('https://services.swpc.noaa.gov/products/alerts.json', timeout=12)
    r.raise_for_status()
    raw['alerts'] = r.json()
    # Filter: ionosphere-relevant messages only
    iono_keywords = ['ionospheric','TEC','HF propagation','radio blackout',
                     'geomagnetic storm','K-index','Kp']
    raw['iono_alerts'] = [
        a for a in raw['alerts']
        if isinstance(a, dict) and
        any(kw.lower() in str(a.get('message','')).lower() for kw in iono_keywords)
    ]
    print(f'  OK NOAA Alerts         {len(raw["alerts"]):>5} total, {len(raw["iono_alerts"])} ionosphere-relevant')
except Exception as e:
    print(f'  ERR NOAA Alerts        {str(e)[:70]}')
    raw['alerts'] = None
    raw['iono_alerts'] = []

# --- 6) Solar wind IMF (Bz – ionospheric coupling) ---
try:
    r = requests.get('https://services.swpc.noaa.gov/json/rtsw/rtsw_mag_1m.json', timeout=12)
    r.raise_for_status()
    raw['imf'] = r.json()
    print(f'  OK IMF/Bz              {len(raw["imf"]):>5} entries')
except Exception as e:
    print(f'  ERR IMF/Bz             {str(e)[:70]}')
    raw['imf'] = None

print('\nData fetch complete')

  OK NOAA Kp               358 entries
  OK GOES X-Ray           2876 entries
  OK F10.7               aktuell 125.69 sfu  (–)
  OK NOAA WWV Bulletin      11 lines
  OK NOAA Alerts           100 total, 53 ionosphere-relevant
  OK IMF/Bz               3660 entries

Data fetch complete


In [4]:
# ============================================================
# PROCESS DATA
# ============================================================

def safe_last(series):
    s = pd.to_numeric(series, errors='coerce').dropna()
    return float(s.iloc[-1]) if not s.empty else None

def kp_strip(s):
    """Removes letter suffixes from Kp strings like '2M', '3+'"""
    m = re.match(r'([0-9]+(?:\.[0-9]*)?)', str(s))
    return float(m.group(1)) if m else None

# --- Kp (last 24h) ---
df_kp = None
kp_now = None
kp_max_24h = None
if raw['kp']:
    df_kp = pd.DataFrame(raw['kp'])
    tc = next((c for c in df_kp.columns if 'time' in c.lower()), df_kp.columns[0])
    kc = 'kp' if 'kp' in df_kp.columns else next(
        (c for c in df_kp.columns if 'kp' in c.lower() and c != tc), df_kp.columns[1])
    df_kp['time'] = pd.to_datetime(df_kp[tc])
    df_kp['kp']   = df_kp[kc].apply(kp_strip)
    df_kp = df_kp[df_kp['kp'] >= 0].dropna(subset=['kp']).sort_values('time').tail(1440)
    if not df_kp.empty:
        kp_now    = float(df_kp['kp'].iloc[-1])
        kp_max_24h= float(df_kp['kp'].max())
        print(f'Kp: current {kp_now:.1f}  max(24h) {kp_max_24h:.1f}')

# --- X-Ray Flux ---
xray_now   = None
xray_class = None
df_xray    = None
if raw['xray']:
    df_xray = pd.DataFrame(raw['xray'])
    tc = next((c for c in df_xray.columns if 'time' in c.lower()), df_xray.columns[0])
    fc = next((c for c in df_xray.columns
               if any(k in c.lower() for k in ['flux','long']) and c != tc),
              df_xray.columns[1])
    df_xray['time'] = pd.to_datetime(df_xray[tc])
    df_xray['flux'] = pd.to_numeric(df_xray[fc], errors='coerce')
    df_xray = df_xray.dropna(subset=['flux']).sort_values('time')
    if not df_xray.empty:
        xray_now   = float(df_xray['flux'].iloc[-1])
        xray_class = ('X' if xray_now >= 1e-4 else 'M' if xray_now >= 1e-5
                      else 'C' if xray_now >= 1e-6 else 'B' if xray_now >= 1e-7 else 'A')
        print(f'X-Ray: {xray_class}-class ({xray_now:.2e} W/m²)')

# --- F10.7 ---
f107_now = None
if raw['f107']:
    try:
        vals = [float(e.get('f10.7', 0)) for e in raw['f107'] if e.get('f10.7')]
        f107_now = vals[-1] if vals else None
        print(f'F10.7: {f107_now:.1f} sfu')
    except: pass

# --- IMF Bz ---
bz_now = None
if raw['imf']:
    df_imf = pd.DataFrame(raw['imf'])
    if 'bz_gsm' in df_imf.columns:
        df_imf['bz_gsm'] = pd.to_numeric(df_imf['bz_gsm'], errors='coerce')
        s = df_imf['bz_gsm'].dropna()
        if not s.empty:
            bz_now = float(s.iloc[-1])
            print(f'IMF Bz: {bz_now:+.1f} nT  ({"southward" if bz_now < -5 else "northward" if bz_now > 0 else "neutral"})')

# --- Parse WWV bulletin ---
wwv_data = {'solar_flux': None, 'a_index': None, 'k_index': None,
            'conditions': [], 'raw': ''}
if raw['wwv']:
    text = raw['wwv']
    wwv_data['raw'] = text[:800]
    # Solar Flux
    m = re.search(r'Solar flux\s+(\d+)', text, re.IGNORECASE)
    if m: wwv_data['solar_flux'] = int(m.group(1))
    # A-Index
    m = re.search(r'\bA[- ]?[Ii]ndex\s+(\d+)', text, re.IGNORECASE)
    if m: wwv_data['a_index'] = int(m.group(1))
    # K-Index
    m = re.search(r'\bK[- ]?[Ii]ndex\s+(\d+)', text, re.IGNORECASE)
    if m: wwv_data['k_index'] = int(m.group(1))
    # Propagation conditions
    for kw in ['normal','disturbed','unsettled','active','storm','quiet']:
        if kw.lower() in text.lower():
            wwv_data['conditions'].append(kw)
    print(f'WWV: Solar Flux={wwv_data["solar_flux"]}  A={wwv_data["a_index"]}  '
          f'K={wwv_data["k_index"]}  Cond={wwv_data["conditions"]}')

# --- Ionospheric warnings ---
if raw.get('iono_alerts'):
    print(f'Ionospheric warnings: {len(raw["iono_alerts"])}')
    for a in raw['iono_alerts'][:3]:
        print(f'  {str(a.get("message",""))[:120]}')

# --- Day/night estimate (local solar time of measurement points) ---
utc_hour = datetime.datetime.utcnow().hour
# Simplified day/night determination for D-Layer
# D-Layer active when local solar time is 06–18h
d_layer_active = (6 <= utc_hour <= 18)  # rough UTC approximation for Central Europe
print(f'\nUTC hour: {utc_hour}  D-Layer active (approximation): {d_layer_active}')
print('Data processing complete')

Kp: current 1.0  max(24h) 2.0


X-Ray: C-class (2.10e-06 W/m²)
F10.7: 125.7 sfu
IMF Bz: -2.1 nT  (neutral)
WWV: Solar Flux=163  A=11  K=None  Cond=['storm']
Ionospheric warnings: 53
  Space Weather Message Code: WARK04
Serial Number: 5370
Issue Time: 2026 Jun 25 2343 UTC

EXTENDED WARNING: Geomagnet
  Space Weather Message Code: ALTK04
Serial Number: 2668
Issue Time: 2026 Jun 25 1941 UTC

ALERT: Geomagnetic K-index 
  Space Weather Message Code: WARK04
Serial Number: 5369
Issue Time: 2026 Jun 25 1939 UTC


UTC hour: 11  D-Layer active (approximation): True
Data processing complete


In [5]:
# ============================================================
# DERIVED VALUES – h_eff, kp_val, f107_val, schumann_freq
# ============================================================

import math

# Schumann resonance frequency formula (geometric, ideal)
# f_n = c * sqrt(n*(n+1)) / (2*pi*(R_earth + h))
R_EARTH_KM = 6371.0

def schumann_freq(n, h_km):
    """Ideal geometric Schumann frequency for mode n at cavity height h_km"""
    c = 299792.458  # km/s
    R = R_EARTH_KM + h_km
    return (c * math.sqrt(n * (n + 1))) / (2 * math.pi * R)

# --- kp_val: current Kp (own data → fallback to Layer 0) ---
kp_val = kp_now if kp_now is not None else l0_kp
if kp_val is None:
    kp_val = 2.0  # neutraler Fallback
    print("⚠️  kp_val: no value available – fallback 2.0")
else:
    kp_val = float(kp_val)

# --- f107_val: current F10.7 (own data → fallback to Layer 0) ---
f107_val = f107_now if f107_now is not None else l0_f107
if f107_val is None:
    f107_val = 120.0  # neutraler Fallback (~solares Minimum/Mittel)
    print("⚠️  f107_val: no value available – fallback 120.0 sfu")
else:
    f107_val = float(f107_val)

# --- h_eff: effective ionospheric cavity height ---
# Physical basis:
#   - Day (D-Layer active): ~70 km
#   - Night (D-Layer dissolved): ~85–90 km
#   - High ionization (F10.7 > 150): layer sinks slightly
#   - Geomagnetic storm (Kp > 5): layer unstable / variable

h_base = 70.0 if d_layer_active else 87.0  # Day vs. night

# Kp correction: high Kp → D/E-Layer lower (more ionization)
kp_correction = -min(kp_val * 0.8, 8.0)  # max -8 km at Kp=10

# F10.7 correction: high radiation → layer slightly lower
f107_correction = -max(0, (f107_val - 120) * 0.03)  # max ~-3 km at F10.7=220

h_eff = round(h_base + kp_correction + f107_correction, 1)
h_eff = max(60.0, min(95.0, h_eff))  # physically constrained

print(f"✅ schumann_freq(1, 80) = {schumann_freq(1, 80):.4f} Hz")
print(f"✅ kp_val   = {kp_val:.1f}")
print(f"✅ f107_val = {f107_val:.1f} sfu")
print(f"✅ h_eff    = {h_eff:.1f} km  (base: {h_base} km | Kp-corr: {kp_correction:+.1f} | F10.7-corr: {f107_correction:+.1f})")


✅ schumann_freq(1, 80) = 10.4599 Hz
✅ kp_val   = 1.0
✅ f107_val = 125.7 sfu
✅ h_eff    = 69.0 km  (base: 70.0 km | Kp-corr: -0.8 | F10.7-corr: -0.2)


---
## 3. Visualizations

In [6]:
# ============================================================
# KP-INDEX + X-RAY FLUX – the two main ionospheric drivers
# ============================================================

n_panels = sum([df_kp is not None, df_xray is not None])
if n_panels > 0:
    titles = []
    if df_kp    is not None: titles.append('Kp-Index (geomagnetic disturbance)')
    if df_xray  is not None: titles.append('GOES X-Ray Flux (Solar UV proxy)')

    fig = make_subplots(rows=n_panels, cols=1, shared_xaxes=True,
                        subplot_titles=titles, vertical_spacing=0.1)
    row = 1

    if df_kp is not None:
        colors_kp = ['#2ecc71' if k < 4 else '#f39c12' if k < 6 else '#e74c3c'
                     for k in df_kp['kp']]
        fig.add_trace(go.Bar(x=df_kp['time'], y=df_kp['kp'],
                             marker_color=colors_kp, opacity=0.8, name='Kp'), row=row, col=1)
        fig.add_hline(y=5, line_dash='dot', line_color='#e74c3c',
                      annotation_text='G1-Storm → Ionosphere disturbed', row=row, col=1)
        fig.add_hline(y=3, line_dash='dot', line_color='#f39c12',
                      annotation_text='Unsettled', row=row, col=1)
        fig.update_yaxes(title_text='Kp [0–9]', range=[0,9], row=row, col=1)
        row += 1

    if df_xray is not None:
        fig.add_trace(go.Scatter(x=df_xray['time'], y=df_xray['flux'],
                                 line=dict(color='#E85D24', width=1.5), name='X-Ray'),
                      row=row, col=1)
        for lvl, lbl, col in [(1e-4,'X','#c0392b'),(1e-5,'M','#e67e22'),
                               (1e-6,'C','#f1c40f'),(1e-7,'B','#2ecc71')]:
            fig.add_hline(y=lvl, line_dash='dot', line_color=col,
                          annotation_text=lbl, row=row, col=1)
        fig.update_yaxes(type='log', title_text='Flux [W/m²]', row=row, col=1)
        row += 1

    fig.update_layout(
        title=dict(text='Ionospheric Main Drivers (last 24h)', font=dict(size=15)),
        height=200 * n_panels + 80,
        showlegend=False, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=70, r=30, t=60, b=40)
    )
    fig.show()

In [7]:
# ============================================================
# RESONANCE CONDITIONS – Schumann frequencies as bar chart
# Each SR mode: frequency at different cavity heights
# ============================================================

modi        = [1, 2, 3, 4]
h_scenarios = {
    'Night / quiet (90 km)':   90.0,
    'Night (85 km)':           85.0,
    'Reference (80 km)':        80.0,
    f'Aktuell ({h_eff:.0f} km)': float(h_eff),
    'Day / active (70 km)':     70.0,
    'Storm / ionized (65 km)': 65.0,
}
colors_h = ['#378ADD','#5F9FDD','#888780','#E85D24','#F2A623','#e74c3c']

fig = go.Figure()

x_labels = [f'SR-{n}' for n in modi]

for i, (label, h) in enumerate(h_scenarios.items()):
    freqs = [schumann_freq(n, h) for n in modi]
    # Highlight if this is the current height
    is_current = abs(h - float(h_eff)) < 0.1
    fig.add_trace(go.Bar(
        name=label,
        x=x_labels,
        y=freqs,
        marker_color=colors_h[i],
        opacity=0.95 if is_current else 0.65,
        marker_line=dict(color='white', width=2) if is_current else dict(width=0),
        text=[f'{f:.2f} Hz' for f in freqs],
        textposition='outside',
        textfont=dict(size=8),
    ))

# Referenz-Linie SR-1 = 7.83 Hz (empirisch)
fig.add_hline(y=7.83, line_dash='dot', line_color='#639922', line_width=1.5,
              annotation_text='Empirical 7.83 Hz (SR-1)',
              annotation_font=dict(size=9, color='#639922'))

fig.update_layout(
    title=dict(text='Schumann Frequencies per Mode – Scenarios by Cavity Height', font=dict(size=14)),
    barmode='group',
    xaxis=dict(title='Schumann Mode'),
    yaxis=dict(title='Frequenz [Hz]', range=[0, max(schumann_freq(4, 65)*1.15, 38)]),
    height=420,
    plot_bgcolor='#1a1a2e',
    paper_bgcolor='rgba(0,0,0,0)',
    legend=dict(title='Cavity Scenario', bgcolor='rgba(20,20,40,0.85)',
                font=dict(color='white', size=10)),
    margin=dict(l=60, r=30, t=55, b=50)
)
fig.show()

# Textausgabe Delta zur Referenz
print(f'\nFrequency delta (current {h_eff:.0f} km vs. reference 80 km):')
ref_f = {n: schumann_freq(n, 80.0) for n in modi}
act_f = {n: schumann_freq(n, float(h_eff)) for n in modi}
for n in modi:
    d = (act_f[n] - ref_f[n]) * 1000
    print(f'  SR-{n}: {act_f[n]:.4f} Hz  (Ref {ref_f[n]:.4f} Hz  Δ {d:+.1f} mHz)')



Frequency delta (current 69 km vs. reference 80 km):
  SR-1: 10.4778 Hz  (Ref 10.4599 Hz  Δ +17.9 mHz)
  SR-2: 18.1481 Hz  (Ref 18.1171 Hz  Δ +30.9 mHz)
  SR-3: 25.6653 Hz  (Ref 25.6215 Hz  Δ +43.8 mHz)
  SR-4: 33.1337 Hz  (Ref 33.0772 Hz  Δ +56.5 mHz)


---
### 📐 Systemic Insight: Geometric Cavity vs. Real Schumann Resonance

**Relevant for:** Layer 6 = delta analysis &nbsp;|&nbsp; Layer 7 = feature vector / system interpretation

A cavity height change from 80 km to 70 km shifts the **idealized geometric frequencies** only minimally:

| Modus | Aktuell | Referenz | Δ |
|-------|---------|----------|---|
| SR-1 | 10.4762 Hz | 10.4599 Hz | +16 mHz |
| SR-2 | 18.1452 Hz | 18.1171 Hz | +28 mHz |
| SR-3 | 25.6612 Hz | 25.6215 Hz | +40 mHz |
| SR-4 | 33.1285 Hz | 33.0772 Hz | +51 mHz |

The geometric effect is limited because 10 km is very small compared to Earth's radius (~6371 km).

**Consequence:** Real Schumann deviations are likely driven more strongly by **conductivity, damping, day/night structure, lightning distribution and source geometry** than by pure cavity height.

---

**Layer 6 compares:**
```
geometric Δ from Layer 4  vs.  observed Δ from real Schumann measurement data
```

**Layer 7 Feature:**
```
If real Δ >> geometric Δ  →  non-geometric factors dominant
```


In [8]:
# ============================================================
# IONOSPHERIC STATE – Summary gauge
# Five dimensions: ionization, disturbance, cavity height, X-Ray, Bz coupling
# ============================================================

def norm(v, lo, hi):
    if v is None: return None
    return round(max(0.0, min(1.0, (v - lo) / (hi - lo))), 3)

# Ionization level (F10.7 → F2-Layer)
ioniz = norm(f107_val, 60, 250)

# Disturbance level (Kp → E/F-Layer irregularities)
disturb = norm(kp_val, 0, 9)

# Cavity height deviation from reference (80 km)
h_dev = norm(abs(h_eff - 80), 0, 20)

# X-Ray ionization (D-Layer absorption)
xray_ioniz = None
if xray_now is not None:
    xray_ioniz = norm(math.log10(max(xray_now, 1e-9)) + 9, 0, 5)

# Bz coupling (southward Bz → stronger magnetosphere-ionosphere coupling)
bz_coupling = norm(-(bz_now or 0), -5, 30) if bz_now is not None else None

dims = [
    ('Ionization Level (F10.7)',    ioniz,      '#F2A623'),
    ('Disturbance Level (Kp)',      disturb,    '#7F77DD'),
    ('Cavity Height Deviation',     h_dev,      '#378ADD'),
    ('X-Ray D-Layer Absorption',    xray_ioniz, '#E85D24'),
    ('IMF Bz Coupling',             bz_coupling,'#534AB7'),
]

fig = go.Figure()
for name, val, col in dims:
    if val is not None:
        h = int(val * 20)
        bar = '█' * h + '░' * (20 - h)
        label = f'{name}:  {bar}  {val:.3f}'
    else:
        label = f'{name}:  {"─" * 20}  n/a'
    fig.add_trace(go.Bar(
        x=[val if val is not None else 0],
        y=[name],
        orientation='h',
        marker_color=col, opacity=0.82,
        showlegend=False
    ))

fig.add_vline(x=0.5, line_dash='dot', line_color='#e74c3c',
              annotation_text='Disturbance Threshold', annotation_position='top')
fig.update_layout(
    title=dict(text='Ionospheric State – 5 Dimensions', font=dict(size=14)),
    xaxis=dict(title='Score [0–1]', range=[0, 1.05]),
    height=330, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=240, r=50, t=55, b=40)
)
fig.show()

print(f'\nIonospheric state:')
for name, val, _ in dims:
    print(f'  {name:<35} {val:.3f}' if val is not None else f'  {name:<35} n/a')


Ionospheric state:
  Ionization Level (F10.7)            0.346
  Disturbance Level (Kp)              0.111
  Cavity Height Deviation             0.550
  X-Ray D-Layer Absorption            0.664
  IMF Bz Coupling                     0.203


In [9]:
# ============================================================
# WWV BULLETIN & IONOSPHERIC WARNINGS
# ============================================================

print('WWV IONOSPHERIC BULLETIN (NOAA)')
print('=' * 60)
if raw.get('wwv'):
    for line in raw['wwv'].splitlines()[:30]:
        if line.strip():
            print(line)
else:
    print('Not available.')

print('\nIONOSPHERE-RELEVANT WARNINGS')
print('=' * 60)
if raw.get('iono_alerts'):
    for a in raw['iono_alerts'][:5]:
        print(str(a.get('message', ''))[:300])
        print('─' * 40)
else:
    print('No ionosphere-relevant warnings.')

WWV IONOSPHERIC BULLETIN (NOAA)
:Product: Geophysical Alert Message wwv.txt
:Issued: 2026 Jun 27 0905 UTC
# Prepared by the US Dept. of Commerce, NOAA, Space Weather Prediction Center
#
#          Geophysical Alert Message
#
Solar-terrestrial indices for 26 June follow.
Solar flux 163 and estimated planetary A-index 11.
The estimated planetary K-index at 0900 UTC on 27 June was 2.00.
No space weather storms were observed for the past 24 hours.
No space weather storms are predicted for the next 24 hours.

IONOSPHERE-RELEVANT WARNINGS
Space Weather Message Code: WARK04
Serial Number: 5370
Issue Time: 2026 Jun 25 2343 UTC

EXTENDED WARNING: Geomagnetic K-index of 4 expected
Extension to Serial Number: 5369
Valid From: 2026 Jun 25 1939 UTC
Now Valid Until: 2026 Jun 26 1200 UTC
Warning Condition: Persistence




NOAA Space We
────────────────────────────────────────
Space Weather Message Code: ALTK04
Serial Number: 2668
Issue Time: 2026 Jun 25 1941 UTC

ALERT: Geomagnetic K-index of 4 
Thre

---
## 4. State Assessment & Handoff to Layer 5

In [10]:
# ============================================================
# LAYER-4-SCORE
# ============================================================

COMPONENTS = {
    'Ionization Level (F10.7)':    {'score': ioniz,       'source': 'primary' if f107_now else 'from_layer0', 'dynamic': True},
    'Disturbance Level (Kp)':          {'score': disturb,     'source': 'primary' if kp_now   else 'from_layer0', 'dynamic': True},
    'Cavity Height Deviation':    {'score': h_dev,       'source': 'derived',                                'dynamic': True},
    'X-Ray Absorption (D-Layer)':{'score': xray_ioniz,  'source': 'primary' if xray_now else 'missing',     'dynamic': True},
    'IMF Bz Coupling':             {'score': bz_coupling, 'source': 'primary' if bz_now   else 'from_layer0', 'dynamic': True},
}

available    = {k: v['score'] for k, v in COMPONENTS.items() if v['score'] is not None}
unavailable  = [k for k, v in COMPONENTS.items() if v['score'] is None]
layer4_score = round(sum(available.values()) / len(available), 4) if available else None
confidence   = round(len(available) / len(COMPONENTS), 2)
level = ('unknown' if layer4_score is None
         else 'quiet'   if layer4_score < 0.3
         else 'moderate' if layer4_score < 0.6
         else 'active')
dominant_l4 = max(available, key=available.get) if available else 'none'

# Resonance delta for Schumann SR-1
sr1_ref = schumann_freq(1, 80.0)
sr1_now = schumann_freq(1, h_eff)
sr1_delta_mhz = (sr1_now - sr1_ref) * 1000

# Propagation conditions
prop_cond = ('disturbed'   if kp_val > 4
             else 'unsettled' if kp_val > 2
             else 'normal')
if xray_class in ['X', 'M']:
    prop_cond = 'Radio-Blackout moeglich'

W = 68
print('=' * W)
print('LAYER 4 – IONOSPHERE – STATE ASSESSMENT')
print('=' * W)
for name, comp in COMPONENTS.items():
    s = comp['score']
    if s is not None:
        bar = '█' * int(s * 20) + '░' * (20 - int(s * 20))
        print(f'  {name:<36} {bar}  {s:.3f}  [{comp["source"]}]')
    else:
        print(f'  {name:<36} {"─" * 20}  n/a   [missing]')
print('-' * W)
print(f'  Score:          {layer4_score:.3f}  ({len(available)}/{len(COMPONENTS)} components)')
print(f'  Confidence:     {confidence:.0%}')
print(f'  Level:          {level.upper()}')
print(f'  Dominant:       {dominant_l4}')
print(f'  Cavity height:  {h_eff:.1f} km  (Ref: 80.0 km)')
print(f'  SR-1:           {sr1_now:.3f} Hz  (Δ {sr1_delta_mhz:+.1f} mHz vs. ref)')
print(f'  Propagation:    {prop_cond}')
print(f'  X-Ray:          {xray_class}-class  ({xray_now:.2e} W/m²)' if xray_now else '  X-Ray:          n/a')
print('=' * W)

# Radar
cats   = list(available.keys())
vals_r = list(available.values())
if len(cats) >= 3:
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=vals_r + [vals_r[0]], theta=cats + [cats[0]],
        fill='toself', fillcolor='rgba(83,74,183,0.20)',
        line=dict(color='#534AB7', width=2.5), name='Layer 4'
    ))
    fig.add_trace(go.Scatterpolar(
        r=[0.5] * (len(cats)+1), theta=cats + [cats[0]],
        line=dict(color='#E24B4A', dash='dot', width=1),
        mode='lines', name='Disturbance Threshold'
    ))
    fig.update_layout(
        title=dict(
            text=f'Layer 4 – Ionospheric Activity Profile | Score: {layer4_score:.3f} | {level.upper()} | Confidence: {confidence:.0%}',
            font=dict(size=12)
        ),
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        height=450, showlegend=True,
        margin=dict(l=80, r=80, t=70, b=40)
    )
    fig.show()

LAYER 4 – IONOSPHERE – STATE ASSESSMENT
  Ionization Level (F10.7)             ██████░░░░░░░░░░░░░░  0.346  [primary]
  Disturbance Level (Kp)               ██░░░░░░░░░░░░░░░░░░  0.111  [primary]
  Cavity Height Deviation              ███████████░░░░░░░░░  0.550  [derived]
  X-Ray Absorption (D-Layer)           █████████████░░░░░░░  0.664  [primary]
  IMF Bz Coupling                      ████░░░░░░░░░░░░░░░░  0.203  [primary]
--------------------------------------------------------------------
  Score:          0.375  (5/5 components)
  Confidence:     100%
  Level:          MODERATE
  Dominant:       X-Ray Absorption (D-Layer)
  Cavity height:  69.0 km  (Ref: 80.0 km)
  SR-1:           10.478 Hz  (Δ +17.9 mHz vs. ref)
  Propagation:    normal
  X-Ray:          C-class  (2.10e-06 W/m²)


In [11]:
# ============================================================
# EXPORT – layer4_test_state.json
# ============================================================

# Schumann downstream
_schumann_l4 = (
    'Cavity height deviation >10 km – SR frequency shift possible'
    if abs(h_eff - 80) > 10
    else f'SR-1 at {sr1_now:.3f} Hz (Δ {sr1_delta_mhz:+.1f} mHz) – minor shift'
)

# GEC downstream (ionosphere as upper electrode)
_gec_l4 = (
    'elevated conductivity through ionization – GEC upper boundary active'
    if ioniz is not None and ioniz > 0.5
    else 'normal ionospheric conductivity'
)

# state_summary
_f107_s  = f'F10.7={f107_now:.0f} sfu.' if f107_now else 'F10.7 n/a.'
_kp_s    = f'Kp={kp_now:.1f} (max 24h: {kp_max_24h:.1f}).' if kp_now is not None else 'Kp n/a.'
_xray_s  = f'X-Ray {xray_class}-Klasse.' if xray_class else 'X-Ray n/a.'
_bz_s    = f'IMF Bz={bz_now:+.1f} nT.' if bz_now is not None else 'IMF Bz n/a.'
_cav_s   = f'Cavity height {h_eff:.1f} km (Ref 80 km, Δ {h_eff-80:+.1f} km).'
_sr_s    = f'SR-1={sr1_now:.3f} Hz (Δ {sr1_delta_mhz:+.1f} mHz).'
_prop_s  = f'Propagation: {prop_cond}.'

state_summary = ' '.join([
    f'Layer-4-State: {level}.',
    _f107_s, _kp_s, _xray_s, _bz_s, _cav_s, _sr_s, _prop_s,
    f'Data completeness: {confidence:.0%}.'
])

layer4_state = {
    'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
    'layer': 4,
    'name':  'Ionosphere',

    'score':      layer4_score,
    'level':      level,
    'confidence': confidence,
    'score_basis': f'{len(available)}/{len(COMPONENTS)} dynamic components',
    'dominant_component': dominant_l4,
    'missing_components': unavailable,

    'components': {
        k: {'score': round(v['score'], 4) if v['score'] is not None else None,
            'source': v['source'], 'dynamic': v['dynamic']}
        for k, v in COMPONENTS.items()
    },

    'raw_values': {
        'F10.7_sfu':          f107_now,
        'Kp_current':         kp_now,
        'Kp_max_24h':         kp_max_24h,
        'xray_flux_Wm2':      xray_now,
        'xray_class':         xray_class,
        'IMF_Bz_nT':          bz_now,
        'wwv_solar_flux':     wwv_data['solar_flux'],
        'wwv_a_index':        wwv_data['a_index'],
        'wwv_k_index':        wwv_data['k_index'],
        'wwv_conditions':     wwv_data['conditions'],
    },

    # Resonanzsystem – das Kernstück dieses Layers
    'resonance_system': {
        'cavity_height_km':     round(h_eff, 1),
        'cavity_height_ref_km': 80.0,
        'cavity_delta_km':      round(h_eff - 80.0, 1),
        'day_night':            'day' if d_layer_active else 'night',
        'd_layer_active':       bool(d_layer_active),
        'schumann_frequencies_Hz': {
            f'SR_{n}': round(schumann_freq(n, h_eff), 4) for n in [1,2,3,4]
        },
        'schumann_ref_Hz': {
            f'SR_{n}': round(schumann_freq(n, 80.0), 4) for n in [1,2,3,4]
        },
        'cavity_delta_insight': {
            'type': 'ideal_geometric_delta',
            'use_case': ['layer6_delta_analysis', 'layer7_feature_vector'],
            'geometric_effect': 'small',
            'reason': '10 km cavity-height change is small relative to Earth radius (~6371 km)',
            'interpretation': (
                'Real Schumann-resonance deviations larger than the geometric delta indicate '
                'dominance of conductivity, damping, day-night structure, '
                'lightning-source distribution or propagation effects.'
            ),
            'layer6_test': 'compare geometric_delta_mHz with observed_delta_mHz',
            'layer7_feature': 'non_geometric_dominance = observed_delta_mHz >> geometric_delta_mHz',
        },
        'schumann_delta_mHz': {
            f'SR_{n}': round((schumann_freq(n, h_eff) - schumann_freq(n, 80.0)) * 1000, 2)
            for n in [1,2,3,4]
        },
        'propagation_conditions': prop_cond,
    },

    'flags': {
        'geomagnetic_storm':    bool(kp_val >= 5),
        'radio_blackout':       bool(xray_class in ['X', 'M']),
        'ionospheric_disturbed':bool(kp_val >= 3 or (xray_class in ['X','M'])),
        'cavity_elevated':      bool(abs(h_eff - 80) > 10),
        'bz_southward_coupled': bool(bz_now is not None and bz_now <= -5),
        'iono_alerts_active':   bool(len(raw.get('iono_alerts', [])) > 0),
    },

    'downstream_expectation': {
        'layer5_gec':  _gec_l4,
        'schumann_resonance': _schumann_l4,
        'layer6_pattern': (
            'Ionosphere structure disturbed – resonance pattern altered'
            if level in ['moderat','aktiv']
            else 'Ionosphere stable – resonance pattern unchanged'
        ),
    },

    'layer_context': {
        'L0': {'score': layer0['score'], 'level': layer0['level'],
               'dominant_driver': layer0.get('dominant_driver')} if layer0 else None,
        'L1': {'score': layer1['score'], 'level': layer1['level']} if layer1 else None,
        'L2': {'score': layer2['score'], 'level': layer2['level'],
               'enso_phase': layer2.get('raw_values',{}).get('ENSO',{}).get('phase_observed')} if layer2 else None,
        'L3': {'score': layer3['score'], 'level': layer3['level'],
               'schumann_potential': l3_schumann,
               'xray_context': l3_xray} if layer3 else None,
    },

    'state_summary': state_summary,
}

# numpy-Typen bereinigen
def _to_python(obj):
    import numpy as np
    if isinstance(obj, dict):  return {k: _to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [_to_python(v) for v in obj]
    if isinstance(obj, np.bool_):    return bool(obj)
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return None if np.isnan(obj) else float(obj)
    return obj
layer4_state = _to_python(layer4_state)

with open('../data/states/layer4_test_state.json', 'w', encoding='utf-8') as f:
    json.dump(layer4_state, f, indent=2, ensure_ascii=False)

print('../data/states/layer4_test_state.json saved')
print(json.dumps(layer4_state, indent=2, ensure_ascii=False))



../data/states/layer4_test_state.json saved
{
  "timestamp": "2026-06-27T11:58:53.758336Z",
  "layer": 4,
  "name": "Ionosphere",
  "score": 0.3748,
  "level": "moderate",
  "confidence": 1.0,
  "score_basis": "5/5 dynamic components",
  "dominant_component": "X-Ray Absorption (D-Layer)",
  "missing_components": [],
  "components": {
    "Ionization Level (F10.7)": {
      "score": 0.346,
      "source": "primary",
      "dynamic": true
    },
    "Disturbance Level (Kp)": {
      "score": 0.111,
      "source": "primary",
      "dynamic": true
    },
    "Cavity Height Deviation": {
      "score": 0.55,
      "source": "derived",
      "dynamic": true
    },
    "X-Ray Absorption (D-Layer)": {
      "score": 0.664,
      "source": "primary",
      "dynamic": true
    },
    "IMF Bz Coupling": {
      "score": 0.203,
      "source": "primary",
      "dynamic": true
    }
  },
  "raw_values": {
    "F10.7_sfu": 125.69,
    "Kp_current": 1.0,
    "Kp_max_24h": 2.0,
    "xray_flux_Wm2": 2

---
## Summary Layer 4

| Aspekt | Inhalt |
|--------|--------|
| **Role** | Upper boundary of the Earth-Ionosphere Cavity |
| **Key quantity** | Effective cavity height h_eff → Schumann frequency shift |
| **Drivers** | Solar UV/F10.7 (ionization), Kp (disturbance), IMF Bz (coupling), X-Ray (D-Layer) |
| **Day/Night** | D-Layer active during day (70 km), dissolved at night (85–90 km) |
| **Data sources** | NOAA SWPC: Kp, X-Ray, F10.7, WWV bulletin, IMF Bz |
| **→ Layer 5** | Ionospheric conductivity → GEC upper boundary |
| **→ Layer 6** | Cavity height + frequency shift → Schumann resonance pattern |
| **Output** | `layer4_test_state.json` with `resonance_system` (cavity height, SR frequencies) |

> **Next step:** `layer5_Global_Electric_Circuit.ipynb` – Global Electric Circuit